<a href="https://colab.research.google.com/github/shahdhesham/Thesis_Set1/blob/main/CodeBlue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Reset all runtime variables
%reset -f


In [ ]:
import os
import glob
import subprocess
import zipfile
from pathlib import Path
from google.colab import files


# CodeBlue

In [ ]:
%cd /content

/content


In [ ]:
!git clone https://github.com/k4black/codebleu.git


fatal: destination path 'codebleu' already exists and is not an empty directory.


In [ ]:
%cd codebleu
!pip install -r requirements.txt


/content/codebleu
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [ ]:
!mkdir -p /content/codebleu/keywords
!curl -o /content/codebleu/keywords/python.txt https://raw.githubusercontent.com/k4black/codebleu/main/keywords/python.txt
!curl -o /content/codebleu/keywords/cpp.txt https://raw.githubusercontent.com/k4black/codebleu/main/keywords/cpp.txt
!curl -o /content/codebleu/keywords/java.txt https://raw.githubusercontent.com/k4black/codebleu/main/keywords/java.txt


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    14  100    14    0     0    105      0 --:--:-- --:--:-- --:--:--   106
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    14  100    14    0     0    106      0 --:--:-- --:--:-- --:--:--   107
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    14  100    14    0     0    100      0 --:--:-- --:--:-- --:--:--   101


In [ ]:
# !pip uninstall -y tree-sitter tree-sitter-python tree-sitter-cpp tree-sitter-java
!pip install git+https://github.com/k4black/codebleu.git#egg=codebleu[all]


DEPRECATION: git+https://github.com/k4black/codebleu.git#egg=codebleu[all] contains an egg fragment with a non-PEP 508 name pip 25.0 will enforce this behaviour change. A possible replacement is to use the req @ url syntax, and remove the egg fragment. Discussion can be found at https://github.com/pypa/pip/issues/11617
  Cloning https://github.com/k4black/codebleu.git to /tmp/pip-install-6r7ax40c/codebleu_94cd0c359ce8465c810b81305caced64
  Running command git clone --filter=blob:none --quiet https://github.com/k4black/codebleu.git /tmp/pip-install-6r7ax40c/codebleu_94cd0c359ce8465c810b81305caced64
  Resolved https://github.com/k4black/codebleu.git to commit b0edb622f6a52fe9d1edc407be5061d3e1462a7f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from codebleu import calc_codebleu

# Code Eval

In [ ]:
# 📥 Upload Generated/Hypothesis C++ ZIP
print("📤 Upload your ZIP file containing GENERATED C++ code:")
uploaded = files.upload()
hyp_zip = next(iter(uploaded))

# 📂 Extract files
print("\n📂 Extracting files...")
with zipfile.ZipFile(hyp_zip, 'r') as zip_ref:
    zip_ref.extractall('hypothesis_folder')

print("✅ Files extracted successfully to 'hypothesis_folder/'")

In [ ]:
# 📍 Step 3: Auto-detect uploaded folder structure
import os

# Reference is hardcoded
ref_base = "reference/selected_100"

# Auto-detect hypothesis folder structure
uploaded_root = "hypothesis_folder"

# Find the actual folder inside the extracted ZIP
folders_in_upload = [f for f in os.listdir(uploaded_root) if os.path.isdir(os.path.join(uploaded_root, f))]

if folders_in_upload:
    # Assuming the structure is: hypothesis_folder/[some_folder]/p001, p002, etc.
    detected_folder = folders_in_upload[0]
    hyp_base = os.path.join(uploaded_root, detected_folder)
    print(f"✅ Auto-detected hypothesis base: {hyp_base}")
else:
    # If files are directly in hypothesis_folder
    hyp_base = uploaded_root
    print(f"✅ Using root as hypothesis base: {hyp_base}")

# Configuration
keywords_dir = Path("/content/codebleu/codebleu/keywords")
custom_weights = (0.1, 0.1, 0.3, 0.5)

# Verify paths
print(f"\n🔍 Verifying paths...")
print(f"Reference base: {ref_base} - Exists: {os.path.exists(ref_base)}")
print(f"Hypothesis base: {hyp_base} - Exists: {os.path.exists(hyp_base)}")

# Show what's inside
if os.path.exists(hyp_base):
    contents = os.listdir(hyp_base)[:10]  # Show first 10 items
    print(f"Contents in hypothesis folder: {contents}")

In [ ]:
# --- Evaluation loop for FLAT structure ---
results = []

# Get all hypothesis .cpp files
hyp_files = sorted(glob.glob(os.path.join(hyp_base, "*.cpp")))

print(f"📊 Found {len(hyp_files)} hypothesis files to evaluate\n")

for hyp_file in hyp_files:
    # Get just the filename (e.g., "p001.cpp")
    filename = os.path.basename(hyp_file)

    # Look for matching reference file with same name
    ref_file = os.path.join(ref_base, filename)

    if not os.path.exists(ref_file):
        print(f"⚠️ Skipping {filename} (No matching reference file found)")
        continue

    # Read hypothesis code
    with open(hyp_file, "r", encoding="utf-8") as h:
        hyp_code = h.read()

    # Read reference code
    with open(ref_file, "r", encoding="utf-8") as r:
        ref_code = r.read().strip()

    if not ref_code:
        print(f"⚠️ Skipping {filename} (Empty reference file)")
        continue

    # Calculate CodeBLEU (comparing hypothesis vs single reference)
    bleu_result = calc_codebleu(
        references=[[ref_code]],  # Note: double brackets for single reference
        predictions=[hyp_code],
        lang="cpp",
        keywords_dir=keywords_dir,
        weights=custom_weights
    )

    results.append({
        "File": filename,
        "Hypothesis": hyp_file,
        "Reference": ref_file,
        "CodeBLEU": bleu_result
    })

    print(f"✓ Evaluated: {filename}")
    if isinstance(bleu_result, dict):
        print(f"   CodeBLEU Score: {bleu_result.get('codebleu', 'N/A'):.4f}")

print(f"\n✅ Evaluation complete! Total files evaluated: {len(results)}")


In [ ]:
# --- Save Results ---
output_file = "evaluation_results.txt"

with open(output_file, "w", encoding="utf-8") as f:
    f.write("🔍 CodeBLEU Evaluation Results\n")
    f.write(f"Total files evaluated: {len(results)}\n\n")
    f.write("=" * 80 + "\n\n")

    for r in results:
        f.write(f"📄 File: {r['File']}\n")
        f.write(f"   Hypothesis: {r['Hypothesis']}\n")
        f.write(f"   Reference:  {r['Reference']}\n")
        f.write("\n   📊 CodeBLEU Scores:\n")

        if isinstance(r['CodeBLEU'], dict):
            for k, v in r['CodeBLEU'].items():
                f.write(f"      {k}: {v:.4f}\n")
        else:
            f.write(f"      {r['CodeBLEU']}\n")

        f.write("\n" + "=" * 80 + "\n\n")

print(f"✅ Results saved to: {output_file}")

In [ ]:

# --- Zip and download ---
zip_name = "evaluation_output.zip"
with zipfile.ZipFile(zip_name, 'w') as zipf:
    zipf.write(output_file)

from google.colab import files
files.download(zip_name)
